# MASD Raw & Exclusion Data with Derived Columns
## Region-aware: Jalna (MH), Ujjain (MP), Meghalaya (ML)

Input CSVs are discovered dynamically from `Inputs/` — any file named like
`<STATE PREFIX> <District>_<Mon-DD-YYYY>_MASD.csv` is picked up and routed to its
region config by the state prefix (MH / MP / ML). The district name and report
date are taken from each file name.

Produces, per region that has input CSVs in `Inputs/`:
  * `<PREFIX>_raw_combined_<date>_MASD.xlsx`  -> full raw data + derived cols (no rows removed)
  * `<PREFIX>_Excluded_<date>.xlsx`           -> only rows flagged with an Exclusion_Reason

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import re
import glob
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

# Hardcoded Paths

In [ ]:
BASE_DIR = r"c:\Users\AayushmanSingh\Desktop\MASD COMBINED"
INPUT_DIR = os.path.join(BASE_DIR, "Inputs")
ROLE_FILE = os.path.join(BASE_DIR, "Role and department sheet.xlsx")
LOCATION_UJJAIN = os.path.join(BASE_DIR, "location_ujjain.xlsx")
LOCATION_MEGHALAYA = os.path.join(BASE_DIR, "Location Meghalaya.xlsx")
LOCATION_JALNA = os.path.join(BASE_DIR, "location jalna.xlsx")

# Region Configurations

In [ ]:
# Region configs are keyed by the state prefix found at the start of each input
# file name. Input files are discovered dynamically at run time, so any file
# named like "MH <District>_<Mon-DD-YYYY>_MASD.csv" is picked up automatically.
REGION_CONFIGS = {
    "MH": {
        "name": "Jalna",
        "state_prefix": "MH",
        "location_master": LOCATION_JALNA,
        "adoption_cols": ["Antenatal care", "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "has_region_column": False,
    },
    "MP": {
        "name": "Ujjain",
        "state_prefix": "MP",
        "location_master": LOCATION_UJJAIN,
        "adoption_cols": ["Antenatal care", "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "has_region_column": False,
    },
    "ML": {
        "name": "Meghalaya",
        "state_prefix": "ML",
        "location_master": LOCATION_MEGHALAYA,
        "adoption_cols": ["First Trimester", "Second Trimester", "Third Trimester",
                          "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "has_region_column": True,
    },
}

# Matches e.g. "MH Jalna_Jul-28-2026_MASD.csv" or "ML East Garo Hills_Jul-28-2026_MASD.csv"
INPUT_FILE_RE = re.compile(
    r"^(?P<prefix>[A-Za-z]{2})\s+(?P<district>.+?)\s*_(?P<date>[A-Za-z]{3}-\d{1,2}-\d{4})_MASD\.csv$",
    re.IGNORECASE,
)


def discover_input_files():
    """Scan Inputs/ and group MASD CSVs by the state prefix in the file name."""
    grouped = {prefix: [] for prefix in REGION_CONFIGS}
    unrecognised = []
    for path in sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv"))):
        m = INPUT_FILE_RE.match(os.path.basename(path))
        prefix = m.group("prefix").upper() if m else None
        if prefix in grouped:
            grouped[prefix].append(path)
        else:
            unrecognised.append(os.path.basename(path))
    for name in unrecognised:
        print(f"Warning: skipping unrecognised input file: {name}")
    return grouped


# Roles that should be flagged (not dropped) with an exclusion reason.
EXCLUDED_ROLES = [
    'Pediatrician + Batch Monitor',
    'Lady Supervisor (AWS) + Batch Monitors',
    'Cuedwell Support',
    'HST Data Analysis Group',
    'HST Data Clerk'
]

# Utility Functions

In [ ]:
def extract_district(file_name):
    """Extract the full district name from an input file name, e.g.
    'ML East Garo Hills_Jul-28-2026_MASD.csv' -> 'East Garo Hills'."""
    base_name = os.path.basename(file_name)
    m = INPUT_FILE_RE.match(base_name)
    if m:
        return m.group("district").strip()
    # Fallback: first underscore token with any leading 2-letter prefix removed.
    first_part = os.path.splitext(base_name)[0].split("_")[0]
    return re.sub(r"^[A-Za-z]{2}\s+", "", first_part).strip()


def clean_location(col):
    return (
        col.astype(str)
        .str.split(",")
        .str[0]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
    )


def process_location_columns_meghalaya(df, master_path):
    """Meghalaya master uses lowercase 'district' column."""
    location_master = pd.read_excel(master_path)
    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Location"])
    location_master = location_master.drop_duplicates(subset=["Location_clean"])
    df = df.merge(
        location_master[["Location_clean", "State", "district", "Actual location", "Block/Taluk"]],
        on="Location_clean",
        how="left"
    )
    df["Actual location"] = df["Actual location"].str.title()
    df["Block/Taluk"] = df["Block/Taluk"].str.title()
    df["District"] = df["District"].str.title()
    df.drop(columns=["Location_clean"], inplace=True)
    return df


def process_location_columns_ujjain(df, master_path):
    """Ujjain master uses capital 'District' column. Preserves filename District on miss."""
    location_master = pd.read_excel(master_path)
    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Location"])
    location_master = location_master.drop_duplicates(subset=["Location_clean"])
    df = df.merge(
        location_master[["Location_clean", "State", "District", "Actual location", "Block/Taluk"]],
        on="Location_clean",
        how="left"
    )
    if "District_x" in df.columns and "District_y" in df.columns:
        df["District"] = df["District_y"].fillna(df["District_x"])
        df.drop(columns=["District_x", "District_y"], inplace=True)
    elif "District" not in df.columns:
        df["District"] = ""

    unmatched_df = df[df["Actual location"].isna()].copy()
    if not unmatched_df.empty:
        print(f"Warning: {len(unmatched_df)} rows could not be matched with location master.")

    for col in ["Actual location", "Block/Taluk", "District"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip().str.title()
    if "Location_clean" in df.columns:
        df.drop(columns=["Location_clean"], inplace=True)
    return df


def process_location_columns_jalna(df, master_path):
    """Jalna master uses columns: 'Cuedwell locat', 'Block name', 'Taluk/Tehsil'."""
    location_master = pd.read_excel(master_path)
    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Cuedwell locat"])
    location_master = location_master.drop_duplicates(subset=["Location_clean"])
    location_master = location_master.rename(columns={
        "Block name": "Actual location",
        "Taluk/Tehsil": "Block/Taluk"
    })
    location_master["State"] = "MH"
    location_master["District"] = "Jalna"
    df = df.merge(
        location_master[["Location_clean", "State", "District", "Actual location", "Block/Taluk"]],
        on="Location_clean",
        how="left"
    )
    if "District_x" in df.columns and "District_y" in df.columns:
        df["District"] = df["District_y"].fillna(df["District_x"])
        df.drop(columns=["District_x", "District_y"], inplace=True)
    elif "District" not in df.columns:
        df["District"] = "Jalna"

    unmatched_df = df[df["Actual location"].isna()].copy()
    if not unmatched_df.empty:
        print(f"Warning: {len(unmatched_df)} rows could not be matched with location master.")

    for col in ["Actual location", "Block/Taluk", "District", "State"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()
            if col != "State":
                df[col] = df[col].str.title()
            else:
                df[col] = df[col].str.upper()
    if "Location_clean" in df.columns:
        df.drop(columns=["Location_clean"], inplace=True)
    return df


def process_location_columns(df, config):
    """Dispatch to the correct location processor for the region."""
    if config["name"] == "Meghalaya":
        return process_location_columns_meghalaya(df, config["location_master"])
    elif config["name"] == "Jalna":
        return process_location_columns_jalna(df, config["location_master"])
    else:
        return process_location_columns_ujjain(df, config["location_master"])


def create_batch_column(df):
    df["Batch_Number"] = (
        df["Training Batch"].astype(str).str.extract(r'Batch\s*(\d+)', expand=False)
    )
    df["Batch"] = None
    mask = df["Batch_Number"].notna()
    df.loc[mask, "Batch"] = (
        df.loc[mask, "District"].astype(str).str.strip() + " B" + df.loc[mask, "Batch_Number"]
    )
    df.drop(columns=["Batch_Number"], inplace=True)
    return df


def apply_role_mapping(df, role_file_path):
    """Merge role mapping and FLAG excluded roles (rows are kept, not dropped)."""
    roles_map = pd.read_excel(role_file_path)
    roles_map["role_clean"] = roles_map["role"].astype(str).str.strip().str.lower()
    df["Role_clean"] = df["Role"].astype(str).str.strip().str.lower()
    df = df.merge(roles_map, left_on="Role_clean", right_on="role_clean", how="left")
    df.rename(columns={
        "Short role": "Roles defined",
        "Role group": "Role group",
        "Department": "Department"
    }, inplace=True)
    df.drop(columns=["Role_clean", "role_clean", "role"], inplace=True, errors="ignore")

    excluded_roles_mask = df['Role'].isin(EXCLUDED_ROLES)
    df.loc[excluded_roles_mask, 'Exclusion_Reason'] = 'Excluded Role'
    return df


def calculate_total_adoptions(df, adoption_cols):
    for col in adoption_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["Total adoptions"] = df[adoption_cols].sum(axis=1)
    return df


def create_total_adoptions_group(df):
    def total_adop_group(value):
        if pd.isna(value) or value < 1:
            return "NO"
        elif 1 <= value < 4:
            return "01_to_03"
        elif 4 <= value < 7:
            return "04_to_06"
        else:
            return "07_or_more"
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Total adoptions group"] = df["Total adoptions"].apply(total_adop_group).astype(str)
    return df


def create_total_adoptions_group_monitoring(df):
    def adop_group(value):
        if pd.isna(value):
            return "NO"
        elif 0 <= value <= 2:
            return "00_to_02"
        elif value == 3:
            return "03"
        else:
            return "04_or_more"
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Total adoptions group_monitoring"] = df["Total adoptions"].apply(adop_group).astype(str)
    return df


def calculate_overall_activities(df):
    cols = ["Antenatal care", "Mother's Protein Intake", "Check growth", "Check BF", "Check CF"]
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["Overall activities"] = df[cols].sum(axis=1)
    return df


def create_overall_activities_group(df):
    def overall_activity_category(x):
        if pd.isna(x) or x == 0:
            return "None"
        elif 1 <= x <= 9:
            return "01 to 09"
        elif 10 <= x <= 19:
            return "10 to 19"
        elif 20 <= x <= 29:
            return "20 to 29"
        elif 30 <= x <= 39:
            return "30 to 39"
        elif 40 <= x <= 49:
            return "40 to 49"
        elif 50 <= x <= 59:
            return "50 to 59"
        elif 60 <= x <= 69:
            return "60 to 69"
        elif 70 <= x <= 79:
            return "70 to 79"
        elif 80 <= x <= 89:
            return "80 to 89"
        elif 90 <= x <= 99:
            return "90 to 99"
        else:
            return "Hundred or more"
    df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
    df["Overall activities group"] = df["Overall activities"].apply(overall_activity_category).astype(str)
    return df


def calculate_percentage_adopted(df, total_cases=10):
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce").fillna(0)
    if total_cases == 0:
        df["Percentage of cases adopted"] = 0
    else:
        df["Percentage of cases adopted"] = ((df["Total adoptions"] * 100) / total_cases).round(2)
    return df


def create_last_activity_group(df):
    df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
    df["Last activity group"] = pd.cut(
        df["No Activity Days"], bins=[-1, 3, 7, float("inf")],
        labels=["03 or less days", "04 to 07 days", "08 or more days"]
    ).astype(object)
    df.loc[df["No Activity Days"].isna(), "Last activity group"] = "No activity"
    return df


def create_last_activity_group_monitoring(df):
    df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
    df["Last activity group_monitoring"] = pd.cut(
        df["No Activity Days"], bins=[-1, 3, 7, float("inf")],
        labels=["03 or less days", "04 to 07 days", "08 or more days"]
    ).astype(str)
    df.loc[df["No Activity Days"].isna(), "Last activity group_monitoring"] = "No activity"
    return df


def create_case_categories(df):
    bins = [-1, 0, 3, 6, 9, float("inf")]
    labels = ["No", "01 to 03", "04 to 06", "07 to 09", "10 or more"]
    category_columns = {
        "Antenatal Care category": "Antenatal care",
        "First Trimester category": "First Trimester",
        "Second Trimester category": "Second Trimester",
        "Third Trimester category": "Third Trimester",
        "Time of Delivery category": "At time of delivery",
        "PNC LTE 5 Months category": "PNC LTE 5 Months",
        "PNC GTE 5 Months category": "PNC GTE 5 Months",
        "Antenatal cases category": "Antenatal care",
        "Total cases category": "Total Active Cases",
        "Child cases category": "Total Child Cases"
    }
    for new_col, base_col in category_columns.items():
        df[base_col] = pd.to_numeric(df[base_col], errors="coerce")
        df[new_col] = pd.cut(df[base_col], bins=bins, labels=labels).astype(str)
    return df


def create_growth_measurement_category(df):
    bins = [-1, 0, 9, 19, 29, 39, 49, float("inf")]
    labels = ["None", "01 to 09", "10 to 19", "20 to 29", "30 to 39", "40 to 49", "50 or more"]
    df["Check growth"] = pd.to_numeric(df["Check growth"], errors="coerce")
    df["Growth Measurement category"] = pd.cut(df["Check growth"], bins=bins, labels=labels).astype(str)
    return df


def create_assess_bf_category(df):
    bins = [-1, 0, 2, 4, 6, 8, 10, float("inf")]
    labels = ["None", "01 to 02", "03 to 04", "05 to 06", "07 to 08", "09 to 10", "11 or more"]
    df["Check BF"] = pd.to_numeric(df["Check BF"], errors="coerce")
    df["Assess BF category"] = pd.cut(df["Check BF"], bins=bins, labels=labels).astype(str)
    return df


def create_assess_cf_category(df):
    bins = [-1, 0, 4, 8, 12, float("inf")]
    labels = ["None", "01 to 04", "05 to 08", "09 to 12", "13 or more"]
    df["Check CF"] = pd.to_numeric(df["Check CF"], errors="coerce")
    df["Assess CF category"] = pd.cut(df["Check CF"], bins=bins, labels=labels).astype(str)
    return df


def create_adoption_group_skill_verify(df):
    bins = [-1, 0, 5, float("inf")]
    labels = ["None", "01 to 05 adoptions", "06 or more adoption"]
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Adoption group for skill verify"] = pd.cut(df["Total adoptions"], bins=bins, labels=labels).astype(str)
    return df


def create_avg_activities_per_adoption(df):
    df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
    df["Total Active Cases"] = pd.to_numeric(df["Total Active Cases"], errors="coerce")
    df["Avg activities/adoption"] = (df["Overall activities"] / df["Total Active Cases"])
    df["Avg activities/adoption"] = df["Avg activities/adoption"].replace([np.inf, -np.inf], np.nan)
    df["Avg activities/adoption"] = df["Avg activities/adoption"].round(2)
    return df


def assign_region(df):
    def get_region(d):
        if pd.isna(d):
            return "Other"
        d = str(d).lower().replace("_", " ").strip()
        if "garo" in d:
            return "Garo"
        elif "khasi" in d:
            return "Khasi"
        elif "jaintia" in d:
            return "Jaintia"
        elif "ri bhoi" in d:
            return "Khasi"
        else:
            return "Other"
    df["Region"] = df["District"].apply(get_region)
    return df


def add_place_of_intervention(df):
    df['Place of intervention'] = df['Role group'].apply(
        lambda x: "Facility" if x == "Staff nurse" else "Community"
    )
    return df


def add_expected_adoption(df):
    df['Expected adoption'] = df['Role group'].apply(lambda x: 10 if x == "Staff nurse" else 6)
    return df


def add_target_adoption_fulfilled(df):
    df['target adoption fulfilled_%'] = (
        df['Total Active Cases'] * 100 / df['Expected adoption']
    ).round(2)
    df['target adoption fulfilled_%'] = df['target adoption fulfilled_%'].fillna(0)
    return df


def format_output_sheet(ws):
    center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
    hdr_font = Font(name='Calibri', size=11, bold=True)
    bdy_font = Font(name='Calibri', size=10)
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column):
        for cell in row:
            cell.alignment = center_align
            cell.font = hdr_font if cell.row == 1 else bdy_font
    for col_cells in ws.columns:
        max_len = max((len(str(c.value)) for c in col_cells if c.value), default=0)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 22)
    ws.freeze_panes = 'A2'

# Main Processing Function (region-aware)

In [ ]:
def process_dataframe(df, file_name, config):
    # --- District Extraction ---
    if file_name:
        df["District"] = extract_district(file_name)

    df['Exclusion_Reason'] = None

    # --- Location Processing ---
    df = process_location_columns(df, config)

    # --- Batch Creation ---
    df = create_batch_column(df)

    # --- Role Mapping (+ flag excluded roles) ---
    df = apply_role_mapping(df, ROLE_FILE)

    # --- Calculations ---
    df = calculate_total_adoptions(df, config["adoption_cols"])
    df = calculate_overall_activities(df)
    df = calculate_percentage_adopted(df, total_cases=10)

    # --- New Columns ---
    df = add_place_of_intervention(df)
    df = add_expected_adoption(df)
    df = add_target_adoption_fulfilled(df)

    # --- Adoption Groups ---
    df = create_total_adoptions_group(df)
    df = create_total_adoptions_group_monitoring(df)

    # --- Overall Activity Group ---
    df = create_overall_activities_group(df)

    # --- Monitoring ---
    df = create_last_activity_group(df)
    df = create_last_activity_group_monitoring(df)

    # --- Case Categories ---
    df = create_case_categories(df)

    # --- Growth / Assessments ---
    df = create_growth_measurement_category(df)
    df = create_assess_bf_category(df)
    df = create_assess_cf_category(df)

    # --- Skill Verify ---
    df = create_adoption_group_skill_verify(df)

    # --- Avg activities/adoption ---
    df = create_avg_activities_per_adoption(df)

    # --- Region Column (Meghalaya only) ---
    if config["has_region_column"]:
        df = assign_region(df)

    # --- Undefined location exclusion flag ---
    df.loc[df['Location'].astype(str).str.lower() == 'undefined', 'Exclusion_Reason'] = 'Undefined Location'

    return df


def get_date_str(file_paths):
    """Extract the date token (e.g. 'Jul-28-2026') from the input file names.
    If the files carry different dates, warn and use the most recent one."""
    dates = {}
    for path in file_paths:
        m = INPUT_FILE_RE.match(os.path.basename(path))
        if not m:
            continue
        token = m.group("date")
        try:
            dates[token] = datetime.strptime(token, "%b-%d-%Y")
        except ValueError:
            pass
    if not dates:
        return "combined"
    if len(dates) > 1:
        print(f"Warning: mixed dates across input files {sorted(dates)}; using most recent.")
    return max(dates, key=dates.get)


def save_formatted(df, output_path):
    df.to_excel(output_path, index=False)
    wb = load_workbook(output_path)
    for ws in wb.worksheets:
        format_output_sheet(ws)
    wb.save(output_path)

# Run All Regions

In [ ]:
def main():
    files_by_prefix = discover_input_files()

    for prefix, config in REGION_CONFIGS.items():
        print(f"\n{'='*60}")
        print(f"Processing: {config['name']} ({prefix})")
        print(f"{'='*60}")

        csv_files = files_by_prefix.get(prefix, [])
        if not csv_files:
            print(f"Skipping {config['name']}: no '{prefix} *_MASD.csv' files found in Inputs/.")
            continue

        raw_dfs = []
        for csv_path in csv_files:
            file_name = os.path.basename(csv_path)
            print(f"Reading: {file_name}  (District: {extract_district(file_name)})")
            df = pd.read_csv(csv_path)
            processed = process_dataframe(df, file_name, config)
            raw_dfs.append(processed)

        combined_raw_df = pd.concat(raw_dfs, ignore_index=True)

        date_str = get_date_str(csv_files)

        # Full raw data with derived columns (no rows removed)
        raw_output_path = os.path.join(BASE_DIR, f"{prefix}_raw_combined_{date_str}_MASD.xlsx")
        save_formatted(combined_raw_df, raw_output_path)
        print(f"Saved raw combined ({len(combined_raw_df)} rows): {raw_output_path}")

        # Excluded records only
        excluded_df = combined_raw_df[combined_raw_df['Exclusion_Reason'].notna()]
        excluded_output_path = os.path.join(BASE_DIR, f"{prefix}_Excluded_{date_str}.xlsx")
        save_formatted(excluded_df, excluded_output_path)
        print(f"Saved excluded ({len(excluded_df)} rows): {excluded_output_path}")

    print("\n" + "=" * 60)
    print("ALL AVAILABLE REGIONS PROCESSED SUCCESSFULLY")
    print("=" * 60)


main()